In [1]:
from pathlib import Path
import numpy as np
import time, threading, queue, io

import imageio.v2 as imageio
from tqdm.auto import tqdm
from astropy.io import fits
from astroquery.mast import Observations

from PIL import Image, ImageDraw
import ipywidgets as widgets
from IPython.display import display

# ---------------------------
# WHAT YOU WANT TO SEE
# ---------------------------
TARGET_NAME = "Saturn"
INSTRUMENT  = "NIRCAM"   # for a real planet image. ("NIRISS"/"NIRSPEC" often look like detector/spectra)
RADIUS      = "0.5 deg"

# Optional: restrict to a specific JWST proposal/program by setting PROGRAM_ID to an int.
# Leaving PROGRAM_ID=None searches all JWST observations near TARGET_NAME.

# Optional: force a specific JWST proposal/program.
PROGRAM_ID = None   # set to an int to restrict to a specific JWST proposal/program

# Prefer products that contain frame sequences first
PREFER_KEYS = ("CALINTS", "RATEINTS", "CAL", "RATE", "I2D")

# Limits
MAX_OBS_TO_TRY   = 30
MAX_FILES        = 5        # download multiple products sequentially
MAX_FRAMES_TOTAL = 400

# Video / preview
FPS_VIDEO   = 12
FPS_PREVIEW = 12
PREVIEW_BUFFER = 180        # loop last N rendered frames while still working

OUT_DIR = Path("jwst_saturn_video")
DATA_DIR = OUT_DIR / "data"
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

video_path = OUT_DIR / f"SATURN_{INSTRUMENT}_preview.mp4"
print("Will write:", video_path.resolve())

Will write: /home/hw1970218/Desktop/fits2/jwst_jupiter_video/JUPITER_NIRCAM_preview.mp4


In [2]:
def to_frame_cube(arr: np.ndarray) -> np.ndarray:
    a = np.asarray(arr)
    if a.ndim == 2:
        return a[None, :, :]
    if a.ndim == 3:
        # typical: (nframes, y, x)
        if a.shape[0] <= 10000:
            return a
        # sometimes: (y, x, nframes)
        if a.shape[2] <= 10000:
            return np.moveaxis(a, 2, 0)
        raise ValueError(f"Unclear 3D layout: {a.shape}")
    if a.ndim == 4:
        # (nints, ngroups, y, x) -> average groups
        return np.nanmean(a, axis=1)
    raise ValueError(f"Unsupported shape: {a.shape}")

def get_obs_table(target, instrument, radius, program_id=None):
    if program_id is None:
        obs = Observations.query_object(target, radius=radius)
    else:
        obs = Observations.query_criteria(obs_collection="JWST", proposal_id=program_id)
        # Narrow to Saturn-like targets if possible
        # (some tables use target_name, some target)
    obs = obs[obs["obs_collection"] == "JWST"]
    inst = np.char.upper(obs["instrument_name"].astype(str))
    obs = obs[np.char.find(inst, instrument.upper()) >= 0]
    return obs

def subgroup_col(tbl):
    for c in tbl.colnames:
        if c.lower() in ("productsubgroupdescription", "productsubgroupdesc"):
            return c
    return None

def pick_products_for_obs(obs_row, prefer_keys=PREFER_KEYS):
    products = Observations.get_product_list(obs_row)
    p = Observations.filter_products(products, productType="SCIENCE", extension="fits")
    if len(p) == 0:
        return None

    sc = subgroup_col(p)
    fn = np.char.lower(p["productFilename"].astype(str))

    for key in prefer_keys:
        if sc and np.any(p[sc] == key):
            return p[p[sc] == key]
        if np.any(np.char.find(fn, key.lower()) >= 0):
            return p[np.char.find(fn, key.lower()) >= 0]

    return p

def download_one(product_row, download_dir):
    manifest = Observations.download_products(product_row[0:1], download_dir=str(download_dir), cache=True)
    local_col = "Local Path" if "Local Path" in manifest.colnames else "local_path"
    return Path(manifest[local_col][0])

def read_frames(fits_path: Path):
    with fits.open(fits_path) as hdul:
        if "SCI" not in hdul:
            raise RuntimeError("No SCI extension.")
        data = hdul["SCI"].data
    return to_frame_cube(data).astype(np.float32)

def background_subtract(frame):
    return frame - np.nanmedian(frame)

def find_disk_bbox(frame, pad=30):
    # Saturn disk is bright: use threshold on a smoothed-ish statistic
    m = np.nan_to_num(frame, nan=np.nanmedian(frame))
    thr = np.nanpercentile(m, 90)  # generous
    mask = m > thr
    if not np.any(mask):
        h,w = m.shape
        return 0,h,0,w
    ys, xs = np.where(mask)
    y0,y1 = ys.min(), ys.max()
    x0,x1 = xs.min(), xs.max()
    h,w = m.shape
    y0 = max(0, y0-pad); y1 = min(h, y1+pad)
    x0 = max(0, x0-pad); x1 = min(w, x1+pad)
    return y0,y1,x0,x1

def robust_vmin_vmax(frames3d):
    vmin, vmax = np.nanpercentile(frames3d, [2, 99.7])
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
        vmin, vmax = np.nanmin(frames3d), np.nanmax(frames3d)
    return float(vmin), float(vmax)

def scale_to_uint8(img, vmin, vmax):
    x = np.nan_to_num(img, nan=vmin)
    x = np.clip((x - vmin) / (vmax - vmin + 1e-12), 0, 1)
    # asinh stretch -> nicer planetary contrast
    x = np.arcsinh(10*x) / np.arcsinh(10)
    return (255*x).astype(np.uint8)

def annotate(rgb, text):
    im = Image.fromarray(rgb)
    draw = ImageDraw.Draw(im)
    draw.rectangle([0, 0, im.size[0], 26], fill=(0,0,0))
    draw.text((6, 5), text, fill=(255,255,255))
    return np.array(im)

def phase_corr_shift(ref, img):
    # FFT phase correlation (pure numpy)
    eps = 1e-9
    F = np.fft.fft2(ref)
    G = np.fft.fft2(img)
    R = F * np.conj(G)
    R /= (np.abs(R) + eps)
    corr = np.abs(np.fft.ifft2(R))
    y, x = np.unravel_index(np.argmax(corr), corr.shape)
    if y > corr.shape[0]//2: y -= corr.shape[0]
    if x > corr.shape[1]//2: x -= corr.shape[1]
    return int(y), int(x)

def align_to_reference(frames3d):
    ref = frames3d[0]
    out = [ref]
    for i in range(1, len(frames3d)):
        dy, dx = phase_corr_shift(ref, frames3d[i])
        out.append(np.roll(frames3d[i], shift=(dy, dx), axis=(0,1)))
    return np.stack(out, axis=0)

def rgb_png_bytes(rgb):
    buf = io.BytesIO()
    Image.fromarray(rgb).save(buf, format="PNG")
    return buf.getvalue()


In [3]:
img_widget = widgets.Image(format="png")
display(img_widget)

stop_flag = threading.Event()
frame_q = queue.Queue(maxsize=500)

def producer():
    try:
        obs = get_obs_table(TARGET_NAME, INSTRUMENT, RADIUS, program_id=PROGRAM_ID)
        if len(obs) == 0:
            raise RuntimeError(
                f"No JWST {INSTRUMENT} observations found for {TARGET_NAME}. "
                "Try PROGRAM_ID=None or switch INSTRUMENT to NIRCAM."
            )

        print("Candidate observations:", len(obs))
        # Try best-looking observations first: many Saturn products have useful titles/targets
        # Just iterate a subset
        frames_written = 0
        files_done = 0

        with imageio.get_writer(video_path, fps=FPS_VIDEO, codec="libx264", quality=8) as writer:
            for idx, o in enumerate(obs[:MAX_OBS_TO_TRY]):
                if files_done >= MAX_FILES or frames_written >= MAX_FRAMES_TOTAL:
                    break

                prods = pick_products_for_obs(o, prefer_keys=PREFER_KEYS)
                if prods is None or len(prods) == 0:
                    continue

                # Download product(s) one by one and process immediately
                # (This is what enables “keep playing what’s rendered so far”)
                for j in range(min(len(prods), MAX_FILES - files_done)):
                    if frames_written >= MAX_FRAMES_TOTAL:
                        break

                    p = prods[j:j+1]
                    fits_path = download_one(p, DATA_DIR)
                    if not fits_path.exists():
                        continue

                    print("Downloaded:", fits_path.name)
                    frames = read_frames(fits_path)

                    # If it's a single 2D image, frames.shape[0]=1; if CALINTS/RATEINTS, you get many frames.
                    # Improve readability:
                    # 1) background-subtract
                    frames = np.array([background_subtract(f) for f in frames], dtype=np.float32)

                    # 2) crop around Saturn using first frame bbox
                    y0,y1,x0,x1 = find_disk_bbox(frames[0], pad=50)
                    frames = frames[:, y0:y1, x0:x1]

                    # 3) optional alignment (helps if Saturn drifts a few pixels)
                    if len(frames) > 2:
                        frames = align_to_reference(frames)

                    # 4) stable scaling per file (you can also compute global across all files)
                    vmin, vmax = robust_vmin_vmax(frames)

                    # Write frames
                    for k in range(len(frames)):
                        if frames_written >= MAX_FRAMES_TOTAL:
                            break
                        u8 = scale_to_uint8(frames[k], vmin, vmax)
                        rgb = np.repeat(u8[..., None], 3, axis=2)
                        rgb = annotate(rgb, f"Saturn | {INSTRUMENT} | frame {frames_written}")
                        writer.append_data(rgb)

                        # push to live preview buffer
                        try:
                            frame_q.put_nowait(rgb_png_bytes(rgb))
                        except queue.Full:
                            pass

                        frames_written += 1

                    files_done += 1

        print("Finished rendering. Total frames:", frames_written)

    finally:
        stop_flag.set()

def consumer():
    buffer = []
    idx = 0
    while not (stop_flag.is_set() and frame_q.empty() and len(buffer) > 0):
        # add new frames to rolling buffer
        while True:
            try:
                b = frame_q.get_nowait()
                buffer.append(b)
                if len(buffer) > PREVIEW_BUFFER:
                    buffer = buffer[-PREVIEW_BUFFER:]
            except queue.Empty:
                break

        # loop what we have so far
        if buffer:
            img_widget.value = buffer[idx % len(buffer)]
            idx += 1

        time.sleep(1.0 / FPS_PREVIEW)

# start threads
t_prod = threading.Thread(target=producer, daemon=True)
t_cons = threading.Thread(target=consumer, daemon=True)
t_prod.start()
t_cons.start()


Image(value=b'')

Candidate observations: 31


/tmp/ipykernel_5836/1511639768.py:110: RuntimeWarning: invalid value encountered in divide
  R /= (np.abs(R) + eps)
IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (165, 503) to (176, 512) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


INFO: Found cached file jwst_jupiter_video/data/mastDownload/JWST/jw01373006001_03102_00003_nrcb2/jw01373006001_03102_00003_nrcb2_rateints.fits with expected size 24649920. [astroquery.query]
Downloaded: jw01373006001_03102_00003_nrcb2_rateints.fits


/tmp/ipykernel_5836/1511639768.py:110: RuntimeWarning: invalid value encountered in divide
  R /= (np.abs(R) + eps)
Exception in thread Thread-5 (producer):
Traceback (most recent call last):
  File "/usr/lib/python3.10/threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.10/threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_5836/1537426131.py", line 68, in producer
  File "/home/hw1970218/.local/lib/python3.10/site-packages/imageio/core/format.py", line 590, in append_data
    return self._append_data(im, total_meta)
  File "/home/hw1970218/.local/lib/python3.10/site-packages/imageio/plugins/ffmpeg.py", line 597, in _append_data
    raise ValueError("All images in a movie should have same size")
ValueError: All images in a movie should have same size


INFO: Found cached file jwst_jupiter_video/data/mastDownload/JWST/jw01373006001_03102_00001_nrcb1/jw01373006001_03102_00001_nrcb1_rateints.fits with expected size 24649920. [astroquery.query]
Downloaded: jw01373006001_03102_00001_nrcb1_rateints.fits


In [4]:
t_prod.join()
t_cons.join()

from IPython.display import Video, display
print("Saved:", video_path.resolve())
display(Video(str(video_path), embed=True))


Saved: /home/hw1970218/Desktop/fits2/jwst_jupiter_video/JUPITER_NIRCAM_preview.mp4
